# Sindhi Tokenizer Benchmark — Stage 1

Evaluates the 4 verified Sindhi-specific tokenizers on 3 test registers:

| Register | Source | Size |
|---|---|---|
| Standardized anchor | FLORES-200 `snd_Arab` (devtest) | ~1012 sentences |
| Formal / encyclopedic | Sindhi Wikipedia (`20231101.sd`) | sampled ~2000 articles |
| Informal / social | Sindhi Sentiment dataset (`alinawazmahar/Sindhi_Sentiment_dataset`) | 4420 sentences |

Tokenizers (Stage 1 — Sindhi-specific only):

1. `aakashMeghwar01/SindhiLM-Tokenizer-v1` (BPE)
2. `aakashMeghwar01/SindhiLM-Tokenizer-v2` (morpheme-aware BPE)
3. `aakashMeghwar01/SindhiLM-Tokenizer-v3` (Unigram, 32K)
4. `fahadqazi/Sindhi-BPE-Tokenizer` (BPE, Twitter-trained)

Metrics per (tokenizer × register): fertility, compression ratio, single-token
retention %, OOV/UNK rate, byte-fallback rate.

**Classical/literary register (Shah Jo Risalo) is deferred.** Three sources
were checked and each hit a real problem: a Mendeley CSV export had its
Sindhi column corrupted to literal `?` characters during an Excel round-trip
(unrecoverable); a Kaggle corpus turned out to be a SQL dump of general text,
not classical poetry; and an Internet Archive OCR scan had real Sindhi script
but too much OCR noise (misrecognized characters, footnote numbers bleeding
into verse text) to use without a cleaning pass. Revisit later — either by
cleaning the OCR text or sourcing a fresh clean file — once Stage 1 is
validated on the three registers below.


In [4]:
import os
print("Top level:", os.listdir("/kaggle/input"))
for entry in os.listdir("/kaggle/input"):
    full = f"/kaggle/input/{entry}"
    if os.path.isdir(full):
        print(f"  {entry}/ ->", os.listdir(full))

Top level: ['datasets']
  datasets/ -> ['alinawaz06']


In [21]:
MODELS_INPUT_DIR = "/kaggle/input/datasets/alinawaz06/models"

In [11]:
# ── Setup ────────────────────────────────────────────────────────────────
!pip install -q datasets transformers openpyxl

import re
import json
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer

pd.set_option("display.max_colwidth", 80)


## 1. Load the four test registers

Each becomes a plain Python list of strings (`List[str]`), one sentence/verse/article-chunk per entry.

In [12]:
# ── 1a. FLORES+ (successor to FLORES-200) Sindhi (snd_Arab) — standardized anchor
# Muennighoff/flores200 used a loading script, and the `datasets` library has
# dropped support for loading scripts entirely — that's why the old call
# raised "Dataset scripts are no longer supported". The actively-maintained
# successor is openlanguagedata/flores_plus (parquet-based), which is gated:
# you need to (1) accept the terms once at
# https://huggingface.co/datasets/openlanguagedata/flores_plus while logged
# in, and (2) provide an HF access token. In Kaggle: Add-ons > Secrets > add
# a secret named HF_TOKEN with your token, then attach it to this notebook.
import os
from huggingface_hub import login

hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
else:
    print("No HF_TOKEN found. If the next line fails with a gated-access "
          "error, add an HF_TOKEN Kaggle secret (see setup steps above).")

flores = load_dataset("openlanguagedata/flores_plus", "snd_Arab", split="devtest")
flores_sentences = flores["text"]
print(f"FLORES+ snd_Arab devtest: {len(flores_sentences)} sentences")
print(flores_sentences[0])

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/227 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/221 [00:00<?, ?it/s]

snd_Arab.jsonl: 0.00B [00:00, ?B/s]

snd_Arab.jsonl: 0.00B [00:00, ?B/s]

Generating dev split:   0%|          | 0/997 [00:00<?, ? examples/s]

Generating devtest split:   0%|          | 0/1012 [00:00<?, ? examples/s]

FLORES+ snd_Arab devtest: 1012 sentences
"اسان وٽ 4 مهيني جو ڪوئو آهي جنهن کي ذيابيطس ناهي جنهن کي ذيابيطس هوندو هو،" هن وڌيڪ چيو.


In [13]:
# ── 1b. Sindhi Wikipedia — formal / encyclopedic ───────────────────────────
wiki = load_dataset("wikimedia/wikipedia", "20231101.sd", split="train")

# Sample ~2000 articles, then split into sentences for a fair per-sentence
# comparison against the other registers (crude splitter — refine later
# with a proper Sindhi/Arabic-script sentence segmenter if needed).
SAMPLE_N = 2000
wiki_sample = wiki.shuffle(seed=42).select(range(min(SAMPLE_N, len(wiki))))

def split_sentences(text):
    # Sindhi uses Arabic-script punctuation; split on ۔ . ! ? and newlines
    parts = re.split(r"[۔\.!؟\n]+", text)
    return [p.strip() for p in parts if len(p.strip()) > 3]

wiki_sentences = []
for row in wiki_sample:
    wiki_sentences.extend(split_sentences(row["text"]))

print(f"Sindhi Wikipedia: {len(wiki_sample)} articles -> {len(wiki_sentences)} sentences")
print(wiki_sentences[0] if wiki_sentences else "(none)")


README.md: 0.00B [00:00, ?B/s]

20231101.sd/train-00000-of-00001.parquet:   0%|          | 0.00/17.6M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16928 [00:00<?, ? examples/s]

Sindhi Wikipedia: 2000 articles -> 33512 sentences
ابن سعد- جو پورو نالو ابو عبدالله محمد -بن- سعد -بن- منيع البصري هو ۽ 168هه/ 784ع ۾ بصري ۾ ڄائو


In [14]:
# ── 1c. Sindhi Sentiment dataset — informal / social register ─────────────
sentiment = load_dataset("alinawazmahar/Sindhi_Sentiment_dataset")
split_name = list(sentiment.keys())[0]
text_col = [c for c in sentiment[split_name].column_names if c.lower() in ("text", "sentence", "review")]
text_col = text_col[0] if text_col else sentiment[split_name].column_names[0]
sentiment_sentences = sentiment[split_name][text_col]
print(f"Sindhi Sentiment dataset: {len(sentiment_sentences)} sentences (column: '{text_col}')")
print(sentiment_sentences[0])


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/227k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3536 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/884 [00:00<?, ? examples/s]

Sindhi Sentiment dataset: 3536 sentences (column: 'text')
سنڌ جو علائقو، خاص ڪري ڪاٺياواڙ، ناقابل اعتبار حد تائين زرخيز مٽي جي جوڙجڪ تي فخر ڪري ٿو.


In [15]:
# ── 1d. Bundle registers ───────────────────────────────────────────────────
REGISTERS = {
    "flores200_anchor": flores_sentences,
    "wikipedia_formal": wiki_sentences,
    "sentiment_informal": list(sentiment_sentences),
}

for name, sents in REGISTERS.items():
    print(f"{name:22s} {len(sents):>6} sentences")


flores200_anchor         1012 sentences
wikipedia_formal        33512 sentences
sentiment_informal       3536 sentences


## 2. Load the four Sindhi-specific tokenizers

In [16]:
TOKENIZER_IDS = {
    "SindhiLM_v1": "aakashMeghwar01/SindhiLM-Tokenizer-v1",
    "SindhiLM_v2": "aakashMeghwar01/SindhiLM-Tokenizer-v2",
    "SindhiLM_v3": "aakashMeghwar01/SindhiLM-Tokenizer-v3",
    "FahadQazi_BPE": "fahadqazi/Sindhi-BPE-Tokenizer",
}

tokenizers = {}
tokenizer_vocab_sizes = {}
for name, repo_id in TOKENIZER_IDS.items():
    try:
        tok = AutoTokenizer.from_pretrained(repo_id)
        tokenizers[name] = tok
        tokenizer_vocab_sizes[name] = tok.vocab_size
        print(f"loaded {name:15s} <- {repo_id}  (vocab_size={tok.vocab_size})")
    except Exception as e:
        print(f"FAILED to load {name} <- {repo_id}: {e}")

tokenizer_config.json:   0%|          | 0.00/246 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.22k [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

loaded SindhiLM_v1     <- aakashMeghwar01/SindhiLM-Tokenizer-v1  (vocab_size=43)


tokenizer_config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

...

loaded SindhiLM_v3     <- aakashMeghwar01/SindhiLM-Tokenizer-v3  (vocab_size=32000)


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

loaded FahadQazi_BPE   <- fahadqazi/Sindhi-BPE-Tokenizer  (vocab_size=13031)


In [48]:
# ── Load the 4 custom-trained Sindhi tokenizers ─────────────────────────────
import sentencepiece as spm

class SPMTokenizerAdapter:
    def __init__(self, model_path):
        self.sp = spm.SentencePieceProcessor(model_file=model_path)
        self.unk_token = "<unk>"

    @property
    def vocab_size(self):
        return self.sp.get_piece_size()

    def encode(self, text, add_special_tokens=False):
        return self.sp.encode(text, out_type=int)

    def convert_ids_to_tokens(self, ids):
        return [self.sp.id_to_piece(i) for i in ids]

    def convert_tokens_to_string(self, tokens):
        return self.sp.decode(tokens)

    def decode(self, ids):
        return self.sp.decode(ids)

    @property
    def vocab_size(self):
        return self.sp.get_piece_size()

    def encode(self, text, add_special_tokens=False):
        return self.sp.encode(text, out_type=int)

    def convert_ids_to_tokens(self, ids):
        return [self.sp.id_to_piece(i) for i in ids]

    def convert_tokens_to_string(self, tokens):
        return self.sp.decode(tokens)

VOCAB_SIZES = [32000, 64000, 100000, 150000]
custom_tokenizer_names = []
for vs in VOCAB_SIZES:
    name = f"CustomSindhi_{vs // 1000}k"
    model_path = f"{MODELS_INPUT_DIR}/sindhi_custom_{vs}.model"
    tok = SPMTokenizerAdapter(model_path)
    tokenizers[name] = tok
    tokenizer_vocab_sizes[name] = tok.vocab_size
    custom_tokenizer_names.append(name)
    print(f"loaded {name:18s} <- {model_path}  (vocab_size={tok.vocab_size})")

loaded CustomSindhi_32k   <- /kaggle/input/datasets/alinawaz06/models/sindhi_custom_32000.model  (vocab_size=32000)
loaded CustomSindhi_64k   <- /kaggle/input/datasets/alinawaz06/models/sindhi_custom_64000.model  (vocab_size=64000)
loaded CustomSindhi_100k  <- /kaggle/input/datasets/alinawaz06/models/sindhi_custom_100000.model  (vocab_size=100000)
loaded CustomSindhi_150k  <- /kaggle/input/datasets/alinawaz06/models/sindhi_custom_150000.model  (vocab_size=150000)


## 3. Metric functions

- **Fertility** — mean tokens per whitespace-delimited word.
- **Compression ratio** — mean characters per token (higher = more compressed).
- **Single-token retention %** — share of words that stay one token.
- **UNK rate** — share of tokens that are the tokenizer's unknown/UNK token.
- **Byte-fallback rate** — share of tokens that look like raw byte-fallback
  artifacts (`<0xNN>`-style tokens some SentencePiece/BPE vocabs emit).


In [24]:
def word_split(text):
    # Sindhi is whitespace-delimited at the word level; this is the
    # denominator for fertility / single-token retention.
    return [w for w in text.strip().split() if w]

def is_byte_fallback_token(tok_str):
    return bool(re.fullmatch(r"<0x[0-9A-Fa-f]{2}>", tok_str)) or tok_str.startswith("\u2581<0x")

def evaluate_tokenizer_on_sentences(tokenizer, sentences, max_sentences=None):
    if max_sentences:
        sentences = sentences[:max_sentences]

    total_words = 0
    total_tokens = 0
    total_chars = 0
    single_token_words = 0
    unk_tokens = 0
    byte_fallback_tokens = 0

    unk_str = tokenizer.unk_token

    for sent in sentences:
        if not sent or not sent.strip():
            continue
        words = word_split(sent)
        total_words += len(words)
        total_chars += len(sent)

        for w in words:
            ids = tokenizer.encode(w, add_special_tokens=False)
            toks = tokenizer.convert_ids_to_tokens(ids)
            n = max(len(ids), 1)
            total_tokens += n
            if n == 1:
                single_token_words += 1
            for t in toks:
                if unk_str is not None and t == unk_str:
                    unk_tokens += 1
                if is_byte_fallback_token(t):
                    byte_fallback_tokens += 1

    if total_words == 0 or total_tokens == 0:
        return None

    return {
        "n_sentences": len(sentences),
        "n_words": total_words,
        "n_tokens": total_tokens,
        "fertility": total_tokens / total_words,
        "compression_chars_per_token": total_chars / total_tokens,
        "single_token_pct": 100 * single_token_words / total_words,
        "unk_rate_pct": 100 * unk_tokens / total_tokens,
        "byte_fallback_pct": 100 * byte_fallback_tokens / total_tokens,
    }


## 4. Run the benchmark

Set `MAX_SENTENCES_PER_REGISTER` to cap runtime while iterating (e.g. 300),
then rerun with `None` for the full pass once the pipeline is verified.


In [25]:
MAX_SENTENCES_PER_REGISTER = None  # set to None for the full run

results = []
for tok_name, tok in tokenizers.items():
    for reg_name, sentences in REGISTERS.items():
        if not sentences:
            print(f"skipping {reg_name} (empty — check data loading in section 1)")
            continue
        metrics = evaluate_tokenizer_on_sentences(tok, sentences, MAX_SENTENCES_PER_REGISTER)
        if metrics is None:
            continue
        metrics["tokenizer"] = tok_name
        metrics["register"] = reg_name
        results.append(metrics)
        print(f"{tok_name:15s} x {reg_name:20s} fertility={metrics['fertility']:.3f}  single_token%={metrics['single_token_pct']:.1f}")

results_df = pd.DataFrame(results)
results_df = results_df[["tokenizer", "register", "n_sentences", "n_words", "n_tokens",
                          "fertility", "compression_chars_per_token", "single_token_pct",
                          "unk_rate_pct", "byte_fallback_pct"]]
results_df


SindhiLM_v1     x flores200_anchor     fertility=4.307  single_token%=0.0
SindhiLM_v1     x wikipedia_formal     fertility=4.297  single_token%=0.2
SindhiLM_v1     x sentiment_informal   fertility=4.258  single_token%=0.1
SindhiLM_v2     x flores200_anchor     fertility=2.617  single_token%=22.3
SindhiLM_v2     x wikipedia_formal     fertility=2.580  single_token%=25.1
SindhiLM_v2     x sentiment_informal   fertility=2.562  single_token%=23.5
SindhiLM_v3     x flores200_anchor     fertility=1.824  single_token%=52.7
SindhiLM_v3     x wikipedia_formal     fertility=1.723  single_token%=55.8
SindhiLM_v3     x sentiment_informal   fertility=1.739  single_token%=55.3
FahadQazi_BPE   x flores200_anchor     fertility=1.940  single_token%=37.7
FahadQazi_BPE   x wikipedia_formal     fertility=2.013  single_token%=36.9
FahadQazi_BPE   x sentiment_informal   fertility=1.831  single_token%=42.2
CustomSindhi_32k x flores200_anchor     fertility=1.233  single_token%=82.4
CustomSindhi_32k x wikipedi

,tokenizer,register,n_sentences,n_words,n_tokens,fertility,compression_chars_per_token,single_token_pct,unk_rate_pct,byte_fallback_pct
0,SindhiLM_v1,flores200_anchor,1012,25356,109217,4.307343,1.094958,0.011832,14.018880,0.000000
1,SindhiLM_v1,wikipedia_formal,33512,453629,1949461,4.297479,1.102844,0.183410,15.137928,0.000000
2,SindhiLM_v1,sentiment_informal,3536,47156,200811,4.258440,1.082047,0.065739,12.541644,0.000000
3,SindhiLM_v2,flores200_anchor,1012,25356,66344,2.616501,1.802544,22.259031,0.000000,0.000000
4,SindhiLM_v2,wikipedia_formal,33512,453629,1170476,2.580249,1.836819,25.052411,0.000000,0.000000
5,SindhiLM_v2,sentiment_informal,3536,47156,120818,2.562092,1.798465,23.545254,0.000000,0.000000
6,SindhiLM_v3,flores200_anchor,1012,25356,46254,1.824184,2.585463,52.705474,4.650409,0.000000
7,SindhiLM_v3,wikipedia_formal,33512,453629,781460,1.722685,2.751199,55.786998,4.998848,0.000000
8,SindhiLM_v3,sentiment_informal,3536,47156,82022,1.739376,2.649131,55.269743,2.940674,0.000000
9,FahadQazi_BPE,flores200_anchor,1012,25356,49200,1.940369,2.430650,37.675501,0.000000,0.000000


In [26]:
results_df["vocab_size"] = results_df["tokenizer"].map(tokenizer_vocab_sizes)
results_df = results_df[["tokenizer", "vocab_size", "register", "n_sentences", "n_words", "n_tokens",
                          "fertility", "compression_chars_per_token", "single_token_pct",
                          "unk_rate_pct", "byte_fallback_pct"]]
results_df.to_csv("sindhi_tokenizer_benchmark_stage1.csv", index=False)
results_df

,tokenizer,vocab_size,register,n_sentences,n_words,n_tokens,fertility,compression_chars_per_token,single_token_pct,unk_rate_pct,byte_fallback_pct
0,SindhiLM_v1,43,flores200_anchor,1012,25356,109217,4.307343,1.094958,0.011832,14.018880,0.000000
1,SindhiLM_v1,43,wikipedia_formal,33512,453629,1949461,4.297479,1.102844,0.183410,15.137928,0.000000
2,SindhiLM_v1,43,sentiment_informal,3536,47156,200811,4.258440,1.082047,0.065739,12.541644,0.000000
3,SindhiLM_v2,151643,flores200_anchor,1012,25356,66344,2.616501,1.802544,22.259031,0.000000,0.000000
4,SindhiLM_v2,151643,wikipedia_formal,33512,453629,1170476,2.580249,1.836819,25.052411,0.000000,0.000000
5,SindhiLM_v2,151643,sentiment_informal,3536,47156,120818,2.562092,1.798465,23.545254,0.000000,0.000000
6,SindhiLM_v3,32000,flores200_anchor,1012,25356,46254,1.824184,2.585463,52.705474,4.650409,0.000000
7,SindhiLM_v3,32000,wikipedia_formal,33512,453629,781460,1.722685,2.751199,55.786998,4.998848,0.000000
8,SindhiLM_v3,32000,sentiment_informal,3536,47156,82022,1.739376,2.649131,55.269743,2.940674,0.000000
9,FahadQazi_BPE,13031,flores200_anchor,1012,25356,49200,1.940369,2.430650,37.675501,0.000000,0.000000


In [27]:
# ── Diagnostic: how does SindhiLM_v1 actually tokenize real Sindhi words? ──
sample_words = ["سنڌي", "ٻولي", "علم", "ڪتاب", "پاڻي", "شهر", "دنيا", "روءِ"]

tok = tokenizers["SindhiLM_v1"]
print(f"unk_token = {tok.unk_token!r}")
print(f"vocab_size = {tok.vocab_size}\n")

for w in sample_words:
    ids = tok.encode(w, add_special_tokens=False)
    toks = tok.convert_ids_to_tokens(ids)
    print(f"{w:10s} -> {toks}  (ids={ids})")

# Also inspect what a genuine byte-fallback / OOV token looks like in this
# vocab's token strings, to check whether the byte_fallback_pct=0.0 result
# is real or a detection-regex miss:
sample_vocab_tokens = tok.convert_ids_to_tokens(list(range(0, 20)))
print("\nFirst 20 vocab tokens:", sample_vocab_tokens)

unk_token = '<unk>'
vocab_size = 43

سنڌي       -> ['▁س', 'ن', 'ڌ', 'ي']  (ids=[18, 6, 33, 7])
ٻولي       -> ['▁', 'ٻ', 'و', 'ل', 'ي']  (ids=[5, 30, 13, 32, 7])
علم        -> ['▁', '<unk>', 'ل', 'م']  (ids=[5, 1, 32, 12])
ڪتاب       -> ['▁ڪتاب']  (ids=[19])
پاڻي       -> ['▁', '<unk>', 'ا', 'ڻ', 'ي']  (ids=[5, 1, 26, 16, 7])
شهر        -> ['▁', 'ش', 'ه', 'ر']  (ids=[5, 35, 8, 11])
دنيا       -> ['▁', 'د', 'ن', 'ي', 'ا']  (ids=[5, 14, 6, 7, 26])
روءِ       -> ['▁', 'ر', 'و', 'ء', '<unk>']  (ids=[5, 11, 13, 36, 1])

First 20 vocab tokens: ['<pad>', '<unk>', '<s>', '</s>', '<mask>', '▁', 'ن', 'ي', 'ه', '.', 'هي', 'ر', 'م', 'و', 'د', 'آ', 'ڻ', 'ک', '▁س', '▁ڪتاب']


## 5. Summary pivot — fertility by tokenizer × register

In [28]:
pivot_fertility = results_df.pivot(index="tokenizer", columns="register", values="fertility").round(3)
pivot_fertility["mean_fertility"] = pivot_fertility.mean(axis=1)
pivot_fertility.sort_values("mean_fertility")


register,flores200_anchor,sentiment_informal,wikipedia_formal,mean_fertility
tokenizer,,,,
CustomSindhi_150k,1.167,1.099,1.158,1.141333
CustomSindhi_100k,1.176,1.103,1.173,1.150667
CustomSindhi_64k,1.191,1.110,1.195,1.165333
CustomSindhi_32k,1.233,1.130,1.251,1.204667
SindhiLM_v3,1.824,1.739,1.723,1.762000
FahadQazi_BPE,1.940,1.831,2.013,1.928000
SindhiLM_v2,2.617,2.562,2.580,2.586333
SindhiLM_v1,4.307,4.258,4.297,4.287333


In [29]:
pivot_single_token = results_df.pivot(index="tokenizer", columns="register", values="single_token_pct").round(1)
pivot_single_token


register,flores200_anchor,sentiment_informal,wikipedia_formal
tokenizer,,,
CustomSindhi_100k,85.5,90.8,87.9
CustomSindhi_150k,85.9,91.0,88.5
CustomSindhi_32k,82.4,89.0,84.0
CustomSindhi_64k,84.7,90.3,86.8
FahadQazi_BPE,37.7,42.2,36.9
SindhiLM_v1,0.0,0.1,0.2
SindhiLM_v2,22.3,23.5,25.1
SindhiLM_v3,52.7,55.3,55.8


## 6. Save results

Written to CSV so this stage's numbers are reproducible before Stage 2
(multilingual tokenizers) gets added.


In [30]:
results_df.to_csv("sindhi_tokenizer_benchmark_stage1.csv", index=False)
print("saved sindhi_tokenizer_benchmark_stage1.csv")


saved sindhi_tokenizer_benchmark_stage1.csv


In [31]:
# ── Qualitative morpheme / category check ──────────────────────────────
# NOTE: this word list is assembled from general Sindhi linguistic
# knowledge, not verified against a dictionary — as the native speaker,
# please sanity-check spelling/diacritics before trusting the comparison,
# and swap in or add words you know are good test cases.

MORPHEME_TEST_WORDS = {
    "genitive_possessive": ["دنيا جو", "خدا جو", "پاڻي جي", "ماڻهن جو"],
    "inflectional_suffixes": ["ڪتابن", "شهرن", "توهانکي", "منهنجو", "هنجي"],
    "aspirated_digraphs": ["ٻار", "ڄاڻ", "ڏينهن", "ڳوٺ", "ڀاءُ", "ٿورو", "ڇوڪرو"],
    "loanwords": ["علم", "تعليم", "دنيا", "اسڪول", "ڪمپيوٽر", "انٽرنيٽ"],
}

rows = []
for category, words in MORPHEME_TEST_WORDS.items():
    for w in words:
        row = {"category": category, "word": w}
        for tok_name, tok in tokenizers.items():
            ids = tok.encode(w, add_special_tokens=False)
            toks = tok.convert_ids_to_tokens(ids)
            row[tok_name] = " | ".join(toks)
        rows.append(row)

morpheme_df = pd.DataFrame(rows)
morpheme_df.to_csv("sindhi_tokenizer_morpheme_check.csv", index=False)
morpheme_df

,category,word,SindhiLM_v1,SindhiLM_v2,SindhiLM_v3,FahadQazi_BPE,CustomSindhi_32k,CustomSindhi_64k,CustomSindhi_100k,CustomSindhi_150k
0,genitive_possessive,دنيا جو,▁ | د | ن | ي | ا | ▁ | ج | و,Ø¯ÙĨÙĬ | Ø§ | ĠØ¬ | ÙĪ,دنيا | ج | و,Ø¯ÙĨÙĬØ§ | ĠØ¬ÙĪ,▁دنيا | ▁جو,▁دنيا | ▁جو,▁دنيا | ▁جو,▁دنيا | ▁جو
1,genitive_possessive,خدا جو,▁ | خ | د | ا | ▁ | ج | و,Ø® | Ø¯Ø§ | ĠØ¬ | ÙĪ,خدا | ج | و,Ø®Ø¯Ø§ | ĠØ¬ÙĪ,▁خدا | ▁جو,▁خدا | ▁جو,▁خدا | ▁جو,▁خدا | ▁جو
2,genitive_possessive,پاڻي جي,▁ | <unk> | ا | ڻ | ي | ▁ | جي,Ù¾ | Ø§ | Ú | » | ÙĬ | ĠØ¬ | ÙĬ,پاڻي | ج | ي,Ù¾Ø§Ú»ÙĬ | ĠØ¬ÙĬ,▁پاڻي | ▁جي,▁پاڻي | ▁جي,▁پاڻي | ▁جي,▁پاڻي | ▁جي
3,genitive_possessive,ماڻهن جو,▁ | ما | ڻ | ه | ن | ▁ | ج | و,ÙħØ§ | Ú | » | ÙĩÙĨ | ĠØ¬ | ÙĪ,ماڻهن | ج | و,ÙħØ§Ú»ÙĩÙĨ | ĠØ¬ÙĪ,▁ماڻهن | ▁جو,▁ماڻهن | ▁جو,▁ماڻهن | ▁جو,▁ماڻهن | ▁جو
4,inflectional_suffixes,ڪتابن,▁ڪتاب | ن,Úª | ØªØ§Ø¨ | ÙĨ,ڪتابن,ÚªØªØ§Ø¨ÙĨ,▁ڪتابن,▁ڪتابن,▁ڪتابن,▁ڪتابن
5,inflectional_suffixes,شهرن,▁ | ش | ه | ر | ن,Ø´ÙĩØ± | ÙĨ,ش | ه | ر | ن,Ø´ | Ùĩ | Ø±ÙĨ,▁شهرن,▁شهرن,▁شهرن,▁شهرن
6,inflectional_suffixes,توهانکي,▁ت | و | ه | ا | ن | ک | ي,ØªÙĪ | Ùĩ | Ø§ÙĨ | Ú© | ÙĬ,ت | و | ه | ا | ن | کي,ØªÙĪÙĩØ§ÙĨ | Ú©ÙĬ,▁توهان | کي,▁توهانکي,▁توهانکي,▁توهانکي
7,inflectional_suffixes,منهنجو,▁ | م | ن | ه | ن | ج | و,ÙħÙĨ | Ùĩ | ÙĨØ¬ | ÙĪ,م | نهن | ج | و,ÙħÙĨÙĩÙĨØ¬ÙĪ,▁منهنجو,▁منهنجو,▁منهنجو,▁منهنجو
8,inflectional_suffixes,هنجي,▁ | ه | ن | جي,Ùĩ | ÙĨØ¬ | ÙĬ,ه | ن | ج | ي,ÙĩÙĨØ¬ÙĬ,▁هن | جي,▁هن | جي,▁هن | جي,▁هن | جي
9,aspirated_digraphs,ٻار,▁ | ٻ | ا | ر,Ù» | Ø§Ø±,ٻار,Ù» | Ø§Ø±,▁ٻار,▁ٻار,▁ٻار,▁ٻار


In [32]:
def decode_tokens_pretty(tokenizer, token_ids):
    """Decode each token id individually and join, so byte-level BPE
    tokenizers (GPT2/RoBERTa-style) show readable Sindhi text instead of
    the raw byte-to-unicode-mapped token strings."""
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    pieces = [tokenizer.convert_tokens_to_string([t]) for t in tokens]
    return " | ".join(pieces)

rows = []
for category, words in MORPHEME_TEST_WORDS.items():
    for w in words:
        row = {"category": category, "word": w}
        for tok_name, tok in tokenizers.items():
            ids = tok.encode(w, add_special_tokens=False)
            row[tok_name] = decode_tokens_pretty(tok, ids)
        rows.append(row)

morpheme_df = pd.DataFrame(rows)
morpheme_df.to_csv("sindhi_tokenizer_morpheme_check.csv", index=False)
morpheme_df

,category,word,SindhiLM_v1,SindhiLM_v2,SindhiLM_v3,FahadQazi_BPE,CustomSindhi_32k,CustomSindhi_64k,CustomSindhi_100k,CustomSindhi_150k
0,genitive_possessive,دنيا جو,▁ | د | ن | ي | ا | ▁ | ج | و,دني | ا | ج | و,دنيا | ج | و,دنيا | جو,دنيا | جو,دنيا | جو,دنيا | جو,دنيا | جو
1,genitive_possessive,خدا جو,▁ | خ | د | ا | ▁ | ج | و,خ | دا | ج | و,خدا | ج | و,خدا | جو,خدا | جو,خدا | جو,خدا | جو,خدا | جو
2,genitive_possessive,پاڻي جي,▁ | <unk> | ا | ڻ | ي | ▁ | جي,پ | ا | � | � | ي | ج | ي,پاڻي | ج | ي,پاڻي | جي,پاڻي | جي,پاڻي | جي,پاڻي | جي,پاڻي | جي
3,genitive_possessive,ماڻهن جو,▁ | ما | ڻ | ه | ن | ▁ | ج | و,ما | � | � | هن | ج | و,ماڻهن | ج | و,ماڻهن | جو,ماڻهن | جو,ماڻهن | جو,ماڻهن | جو,ماڻهن | جو
4,inflectional_suffixes,ڪتابن,▁ڪتاب | ن,ڪ | تاب | ن,ڪتابن,ڪتابن,ڪتابن,ڪتابن,ڪتابن,ڪتابن
5,inflectional_suffixes,شهرن,▁ | ش | ه | ر | ن,شهر | ن,ش | ه | ر | ن,ش | ه | رن,شهرن,شهرن,شهرن,شهرن
6,inflectional_suffixes,توهانکي,▁ت | و | ه | ا | ن | ک | ي,تو | ه | ان | ک | ي,ت | و | ه | ا | ن | کي,توهان | کي,توهان | کي,توهانکي,توهانکي,توهانکي
7,inflectional_suffixes,منهنجو,▁ | م | ن | ه | ن | ج | و,من | ه | نج | و,م | نهن | ج | و,منهنجو,منهنجو,منهنجو,منهنجو,منهنجو
8,inflectional_suffixes,هنجي,▁ | ه | ن | جي,ه | نج | ي,ه | ن | ج | ي,هنجي,هن | جي,هن | جي,هن | جي,هن | جي
9,aspirated_digraphs,ٻار,▁ | ٻ | ا | ر,ٻ | ار,ٻار,ٻ | ار,ٻار,ٻار,ٻار,ٻار


## Next steps

Once these three registers load cleanly and the numbers look sane:

1. Revisit the classical/literary register (Shah Jo Risalo) — either clean
   the Internet Archive OCR text (strip footnote-number tokens, repeated page
   headers, garbled front-matter) or source a fresh clean file, then add it
   as a fourth register with the same `evaluate_tokenizer_on_sentences` call.
2. Add the multilingual tokenizers — `FacebookAI/xlm-roberta-base`, `google/mt5-base`,
   `Qwen/Qwen2.5-0.5B`, `bigscience/bloom-560m` (all ungated). Handle
   `google/gemma-3-1b-pt` and `meta-llama/Llama-3.2-1B` separately since both
   are gated and need HF access approval first.
3. Add `google/byt5-small` as a separate "Tier C" baseline — report it in its
   own table, not blended into the subword-tokenizer fertility comparison
   (byte-level fertility numbers aren't comparable to subword fertility).
4. Add the qualitative morpheme checks (izafat construction, aspirated
   digraphs, common inflectional suffixes) as a hand-scored spot-check on
   ~30–50 words, separate from the aggregate metrics above.


## 8. Stage 2 — Multilingual tokenizers

In [33]:
# ── 8a. Ungated multilingual tokenizers ────────────────────────────────────
TOKENIZER_IDS_STAGE2 = {
    "XLM-RoBERTa": "FacebookAI/xlm-roberta-base",
    "mT5": "google/mt5-base",
    "Qwen2.5": "Qwen/Qwen2.5-0.5B",
    "BLOOM": "bigscience/bloom-560m",
}

for name, repo_id in TOKENIZER_IDS_STAGE2.items():
    try:
        tok = AutoTokenizer.from_pretrained(repo_id)
        tokenizers[name] = tok
        tokenizer_vocab_sizes[name] = tok.vocab_size
        print(f"loaded {name:15s} <- {repo_id}  (vocab_size={tok.vocab_size})")
    except Exception as e:
        print(f"FAILED to load {name} <- {repo_id}: {e}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

loaded XLM-RoBERTa     <- FacebookAI/xlm-roberta-base  (vocab_size=250002)


...

config.json:   0%|          | 0.00/693 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

loaded BLOOM           <- bigscience/bloom-560m  (vocab_size=250680)


In [34]:
# ── 8b. Gated multilingual tokenizers ──────────────────────────────────────
# Both require: (1) requesting access on the model's HF page while logged in
# (approval isn't always instant), and (2) the HF_TOKEN secret from section 1a.
TOKENIZER_IDS_GATED = {
    "Gemma3": "google/gemma-3-1b-pt",
    "Llama3.2": "meta-llama/Llama-3.2-1B",
}

for name, repo_id in TOKENIZER_IDS_GATED.items():
    try:
        tok = AutoTokenizer.from_pretrained(repo_id, token=hf_token)
        tokenizers[name] = tok
        tokenizer_vocab_sizes[name] = tok.vocab_size
        print(f"loaded {name:15s} <- {repo_id}  (vocab_size={tok.vocab_size})")
    except Exception as e:
        print(f"FAILED to load {name} <- {repo_id}: {e}  "
              f"(check you've requested + been granted access on the model page)")

config.json:   0%|          | 0.00/880 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

loaded Llama3.2        <- meta-llama/Llama-3.2-1B  (vocab_size=128000)


In [35]:
# ── 8c. Run Stage 2 tokenizers on the same 3 registers ─────────────────────
stage2_names = list(TOKENIZER_IDS_STAGE2.keys()) + list(TOKENIZER_IDS_GATED.keys())

stage2_results = []
for tok_name in stage2_names:
    if tok_name not in tokenizers:
        continue  # failed to load above
    tok = tokenizers[tok_name]
    for reg_name, sentences in REGISTERS.items():
        metrics = evaluate_tokenizer_on_sentences(tok, sentences, MAX_SENTENCES_PER_REGISTER)
        if metrics is None:
            continue
        metrics["tokenizer"] = tok_name
        metrics["register"] = reg_name
        metrics["vocab_size"] = tokenizer_vocab_sizes.get(tok_name)
        stage2_results.append(metrics)
        print(f"{tok_name:15s} x {reg_name:20s} fertility={metrics['fertility']:.3f}")

stage2_df = pd.DataFrame(stage2_results)
results_df = pd.concat([results_df, stage2_df], ignore_index=True)
results_df.to_csv("sindhi_tokenizer_benchmark_stage1_and_2.csv", index=False)
results_df

XLM-RoBERTa     x flores200_anchor     fertility=1.552
XLM-RoBERTa     x wikipedia_formal     fertility=1.601
XLM-RoBERTa     x sentiment_informal   fertility=1.434
mT5             x flores200_anchor     fertility=2.318
mT5             x wikipedia_formal     fertility=2.293
mT5             x sentiment_informal   fertility=2.259
Qwen2.5         x flores200_anchor     fertility=2.617
Qwen2.5         x wikipedia_formal     fertility=2.580
Qwen2.5         x sentiment_informal   fertility=2.562
BLOOM           x flores200_anchor     fertility=2.692
BLOOM           x wikipedia_formal     fertility=2.576
BLOOM           x sentiment_informal   fertility=2.645
Gemma3          x flores200_anchor     fertility=2.341
Gemma3          x wikipedia_formal     fertility=2.331
Gemma3          x sentiment_informal   fertility=2.276
Llama3.2        x flores200_anchor     fertility=2.951
Llama3.2        x wikipedia_formal     fertility=2.867
Llama3.2        x sentiment_informal   fertility=2.907


,tokenizer,vocab_size,register,n_sentences,n_words,n_tokens,fertility,compression_chars_per_token,single_token_pct,unk_rate_pct,byte_fallback_pct
0,SindhiLM_v1,43,flores200_anchor,1012,25356,109217,4.307343,1.094958,0.011832,14.018880,0.000000
1,SindhiLM_v1,43,wikipedia_formal,33512,453629,1949461,4.297479,1.102844,0.183410,15.137928,0.000000
2,SindhiLM_v1,43,sentiment_informal,3536,47156,200811,4.258440,1.082047,0.065739,12.541644,0.000000
3,SindhiLM_v2,151643,flores200_anchor,1012,25356,66344,2.616501,1.802544,22.259031,0.000000,0.000000
4,SindhiLM_v2,151643,wikipedia_formal,33512,453629,1170476,2.580249,1.836819,25.052411,0.000000,0.000000
5,SindhiLM_v2,151643,sentiment_informal,3536,47156,120818,2.562092,1.798465,23.545254,0.000000,0.000000
6,SindhiLM_v3,32000,flores200_anchor,1012,25356,46254,1.824184,2.585463,52.705474,4.650409,0.000000
7,SindhiLM_v3,32000,wikipedia_formal,33512,453629,781460,1.722685,2.751199,55.786998,4.998848,0.000000
8,SindhiLM_v3,32000,sentiment_informal,3536,47156,82022,1.739376,2.649131,55.269743,2.940674,0.000000
9,FahadQazi_BPE,13031,flores200_anchor,1012,25356,49200,1.940369,2.430650,37.675501,0.000000,0.000000


## 9. Tier C — byte-level baseline (ByT5)

Reported separately, never merged into the table above: byte-level fertility
(tokens/word for a tokenizer where every input character becomes multiple
tokens by design) isn't comparable to subword fertility, and blending it in
would silently make ByT5 look artificially terrible without meaning anything.

In [38]:
# ── 9. ByT5 — separate Tier C table ─────────────────────────────────────────
byt5_tok = AutoTokenizer.from_pretrained("google/byt5-small")

byt5_results = []
for reg_name, sentences in REGISTERS.items():
    metrics = evaluate_tokenizer_on_sentences(byt5_tok, sentences, MAX_SENTENCES_PER_REGISTER)
    if metrics is None:
        continue
    metrics["tokenizer"] = "ByT5-small"
    metrics["register"] = reg_name
    byt5_results.append(metrics)

byt5_df = pd.DataFrame(byt5_results)
byt5_df.to_csv("sindhi_tokenizer_benchmark_byt5_tierC.csv", index=False)
byt5_df

,n_sentences,n_words,n_tokens,fertility,compression_chars_per_token,single_token_pct,unk_rate_pct,byte_fallback_pct,tokenizer,register
0,1012,25356,187532,7.395962,0.637694,0.149866,0.0,0.0,ByT5-small,flores200_anchor
1,33512,453629,3376520,7.443351,0.636736,0.499086,0.0,0.0,ByT5-small,wikipedia_formal
2,3536,47156,344157,7.298265,0.631360,0.146323,0.0,0.0,ByT5-small,sentiment_informal


In [39]:
# ── Diagnostic: is SindhiLM_v2 literally the Qwen2.5 tokenizer? ────────────
import hashlib

def file_hash(tokenizer):
    """Hash the tokenizer's vocabulary content to compare two tokenizers
    directly, independent of repo name/config metadata."""
    vocab = tokenizer.get_vocab()
    # sort for a stable, order-independent hash
    serialized = str(sorted(vocab.items())).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()

v2_tok = tokenizers["SindhiLM_v2"]
qwen_tok = tokenizers["Qwen2.5"]

print(f"SindhiLM_v2 vocab_size: {v2_tok.vocab_size}")
print(f"Qwen2.5     vocab_size: {qwen_tok.vocab_size}")
print(f"SindhiLM_v2 vocab hash: {file_hash(v2_tok)}")
print(f"Qwen2.5     vocab hash: {file_hash(qwen_tok)}")
print(f"Identical vocab: {file_hash(v2_tok) == file_hash(qwen_tok)}\n")

# Also compare a few specific token IDs directly
test_word = "دنيا"
print(f"'{test_word}' via SindhiLM_v2: {v2_tok.encode(test_word, add_special_tokens=False)}")
print(f"'{test_word}' via Qwen2.5:     {qwen_tok.encode(test_word, add_special_tokens=False)}")

# Check the special tokens / config too — a relabeled repo often keeps
# identical special token IDs and chat template metadata
print(f"\nSindhiLM_v2 special tokens: {v2_tok.special_tokens_map}")
print(f"Qwen2.5     special tokens: {qwen_tok.special_tokens_map}")
print(f"\nSindhiLM_v2 name_or_path: {v2_tok.name_or_path}")
print(f"Qwen2.5     name_or_path: {qwen_tok.name_or_path}")

SindhiLM_v2 vocab_size: 151643
Qwen2.5     vocab_size: 151643
SindhiLM_v2 vocab hash: e51df1af2d409732b3640867f8fd5499ae4c7ef2cab2d007f2d7894adbaf0ba7
Qwen2.5     vocab hash: e51df1af2d409732b3640867f8fd5499ae4c7ef2cab2d007f2d7894adbaf0ba7
Identical vocab: True

'دنيا' via SindhiLM_v2: [127650, 5703]
'دنيا' via Qwen2.5:     [127650, 5703]

SindhiLM_v2 special tokens: {'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}
Qwen2.5     special tokens: {'eos_token': '<|endoftext|>', 'pad_token': '<|endoftext|>'}

SindhiLM_v2 name_or_path: aakashMeghwar01/SindhiLM-Tokenizer-v2
Qwen2.5     name_or_path: Qwen/Qwen2.5-0.5B


In [40]:
# ── Tokenizer audit: vocab size, special tokens, name_or_path, dedup check ──
import hashlib
import pandas as pd

def vocab_hash(tokenizer):
    vocab = tokenizer.get_vocab()
    serialized = str(sorted(vocab.items())).encode("utf-8")
    return hashlib.sha256(serialized).hexdigest()

audit_rows = []
hashes = {}
for name, tok in tokenizers.items():
    h = vocab_hash(tok)
    hashes[name] = h
    audit_rows.append({
        "tokenizer": name,
        "repo_id": tok.name_or_path,
        "vocab_size": tok.vocab_size,
        "eos_token": tok.special_tokens_map.get("eos_token"),
        "pad_token": tok.special_tokens_map.get("pad_token"),
        "unk_token": tok.special_tokens_map.get("unk_token"),
        "vocab_hash_short": h[:12],
    })

tokenizer_audit_df = pd.DataFrame(audit_rows)
print(tokenizer_audit_df.to_string(index=False))

# ── Duplicate sweep: check every pair, not just the one we already found ──
print("\n--- Duplicate vocab check (all pairs) ---")
names = list(hashes.keys())
found_any = False
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        if hashes[names[i]] == hashes[names[j]]:
            print(f"IDENTICAL VOCAB: {names[i]}  ==  {names[j]}")
            found_any = True
if not found_any:
    print("No other duplicate vocabularies found.")

tokenizer_audit_df.to_csv("tokenizer_audit.csv", index=False)

AttributeError: 'SPMTokenizerAdapter' object has no attribute 'get_vocab'

In [41]:
# ── Dataset audit: source, split, and sanity stats for every register ──────
import statistics

register_sources = {
    "flores200_anchor": "openlanguagedata/flores_plus (config: snd_Arab, split: devtest)",
    "wikipedia_formal": "wikimedia/wikipedia (config: 20231101.sd, ~2000 articles sampled, sentence-split)",
    "sentiment_informal": "alinawazmahar/Sindhi_Sentiment_dataset",
}

dataset_audit_rows = []
for reg_name, sentences in REGISTERS.items():
    word_counts = [len(word_split(s)) for s in sentences if s.strip()]
    char_counts = [len(s) for s in sentences if s.strip()]
    dataset_audit_rows.append({
        "register": reg_name,
        "source": register_sources.get(reg_name, "unknown"),
        "n_sentences": len(sentences),
        "total_words": sum(word_counts),
        "total_chars": sum(char_counts),
        "avg_words_per_sentence": round(statistics.mean(word_counts), 2) if word_counts else 0,
        "min_words_per_sentence": min(word_counts) if word_counts else 0,
        "max_words_per_sentence": max(word_counts) if word_counts else 0,
        "empty_or_blank_entries": sum(1 for s in sentences if not s.strip()),
    })

dataset_audit_df = pd.DataFrame(dataset_audit_rows)
print(dataset_audit_df.to_string(index=False))

# Flag anything that looks off: e.g. Wikipedia's sentence splitter is a crude
# regex — a suspiciously low avg_words_per_sentence or a high min of 0-1
# words would indicate it's producing junk fragments, not real sentences.
for _, row in dataset_audit_df.iterrows():
    if row["avg_words_per_sentence"] < 3:
        print(f"\n⚠ {row['register']}: avg {row['avg_words_per_sentence']} words/sentence "
              f"looks low — inspect the splitter output before trusting this register's numbers.")

dataset_audit_df.to_csv("dataset_audit.csv", index=False)

          register                                                                            source  n_sentences  total_words  total_chars  avg_words_per_sentence  min_words_per_sentence  max_words_per_sentence  empty_or_blank_entries
  flores200_anchor                   openlanguagedata/flores_plus (config: snd_Arab, split: devtest)         1012        25356       119588                   25.06                       8                      66                       0
  wikipedia_formal wikimedia/wikipedia (config: 20231101.sd, ~2000 articles sampled, sentence-split)        33512       453629      2149952                   13.54                       1                     408                       0
sentiment_informal                                            alinawazmahar/Sindhi_Sentiment_dataset         3536        47156       217287                   13.34                       2                      54                       0


In [42]:
# ── Inspect and clean Wikipedia sentence-splitter extremes ────────────────
short_frags = [s for s in wiki_sentences if len(word_split(s)) <= 2]
long_blobs = [s for s in wiki_sentences if len(word_split(s)) >= 100]

print(f"Fragments (<=2 words): {len(short_frags)}  ({100*len(short_frags)/len(wiki_sentences):.1f}%)")
print(f"Blobs (>=100 words):   {len(long_blobs)}  ({100*len(long_blobs)/len(wiki_sentences):.1f}%)")
print("\nSample fragments:", short_frags[:5])
print("\nSample blob (truncated):", long_blobs[0][:300] if long_blobs else "none")

# Filter to a plausible real-sentence range before re-running the benchmark
wiki_sentences_clean = [s for s in wiki_sentences if 3 <= len(word_split(s)) <= 100]
print(f"\nOriginal: {len(wiki_sentences)}  ->  Cleaned: {len(wiki_sentences_clean)} "
      f"({100*len(wiki_sentences_clean)/len(wiki_sentences):.1f}% kept)")

REGISTERS["wikipedia_formal"] = wiki_sentences_clean

Fragments (<=2 words): 7975  (23.8%)
Blobs (>=100 words):   60  (0.2%)

Sample fragments: ['حوالا', 'حوالا', 'حوالا', 'پڻ ڏسو', 'انب جوڳي']

Sample blob (truncated): ان کان سواءِ ملڪ جي ڪيترن ئي علمي، ادبي، تحقيقي ۽ تاريخي ادارن جو ميمبر رهيو، جهڙوڪ: اٿارٽي فار موهن جو دڙو، انٽر يونيورسٽي بورڊ آف پاڪستان، چيئرمئن وائيس چانسلرس ڪميٽي آف يونيورسٽي گرانٽس ڪميشن، ميمبر پاڪستان هسٽاريڪل رڪارڊ اينڊ آرڪائيوز ڪميشن، ميمبر سينٽرل اردو بورڊ لاهور، ميمبر سنڌي ادبي بورڊ، مي

Original: 33512  ->  Cleaned: 25480 (76.0% kept)


In [43]:
# ── Re-run wikipedia_formal only, across every tokenizer, and patch results ─
wiki_clean_results = []
for tok_name, tok in tokenizers.items():
    metrics = evaluate_tokenizer_on_sentences(tok, wiki_sentences_clean, MAX_SENTENCES_PER_REGISTER)
    if metrics is None:
        continue
    metrics["tokenizer"] = tok_name
    metrics["register"] = "wikipedia_formal"
    metrics["vocab_size"] = tokenizer_vocab_sizes.get(tok_name)
    wiki_clean_results.append(metrics)
    print(f"{tok_name:15s} fertility={metrics['fertility']:.3f}  single_token%={metrics['single_token_pct']:.1f}")

wiki_clean_df = pd.DataFrame(wiki_clean_results)

# Drop the old (junk-inflated) wikipedia_formal rows from results_df, replace with clean ones
results_df = results_df[results_df["register"] != "wikipedia_formal"]
results_df = pd.concat([results_df, wiki_clean_df], ignore_index=True)

# Same for ByT5's separate Tier C table
byt5_wiki_metrics = evaluate_tokenizer_on_sentences(byt5_tok, wiki_sentences_clean, MAX_SENTENCES_PER_REGISTER)
byt5_wiki_metrics["tokenizer"] = "ByT5-small"
byt5_wiki_metrics["register"] = "wikipedia_formal"
byt5_df = byt5_df[byt5_df["register"] != "wikipedia_formal"]
byt5_df = pd.concat([byt5_df, pd.DataFrame([byt5_wiki_metrics])], ignore_index=True)

results_df.to_csv("sindhi_tokenizer_benchmark_final.csv", index=False)
byt5_df.to_csv("sindhi_tokenizer_benchmark_byt5_tierC_final.csv", index=False)
print("\nresults_df and byt5_df updated with cleaned wikipedia_formal.")

SindhiLM_v1     fertility=4.271  single_token%=0.2
SindhiLM_v2     fertility=2.568  single_token%=25.5
SindhiLM_v3     fertility=1.726  single_token%=55.4
FahadQazi_BPE   fertility=2.002  single_token%=37.4
CustomSindhi_32k fertility=1.249  single_token%=84.0
CustomSindhi_64k fertility=1.194  single_token%=86.9
CustomSindhi_100k fertility=1.171  single_token%=88.0
CustomSindhi_150k fertility=1.157  single_token%=88.6
XLM-RoBERTa     fertility=1.589  single_token%=63.1
mT5             fertility=2.285  single_token%=22.1
Qwen2.5         fertility=2.568  single_token%=25.5
BLOOM           fertility=2.569  single_token%=25.3
Gemma3          fertility=2.317  single_token%=28.0
Llama3.2        fertility=2.856  single_token%=18.5

results_df and byt5_df updated with cleaned wikipedia_formal.


In [44]:
# ── Spot-check FLORES+ and Sentiment for the same fragment/blob artifacts ──
def audit_register(name, sentences, frag_threshold=2, blob_threshold=100):
    frags = [s for s in sentences if len(word_split(s)) <= frag_threshold]
    blobs = [s for s in sentences if len(word_split(s)) >= blob_threshold]
    print(f"--- {name} ---")
    print(f"Total: {len(sentences)}")
    print(f"Fragments (<={frag_threshold} words): {len(frags)}  ({100*len(frags)/len(sentences):.1f}%)")
    print(f"Blobs (>={blob_threshold} words):   {len(blobs)}  ({100*len(blobs)/len(sentences):.1f}%)")
    if frags:
        print("Sample fragments:", frags[:5])
    if blobs:
        print("Sample blob (truncated):", blobs[0][:300])
    print()
    return frags, blobs

flores_frags, flores_blobs = audit_register("flores200_anchor", flores_sentences)
sentiment_frags, sentiment_blobs = audit_register("sentiment_informal", list(sentiment_sentences))

--- flores200_anchor ---
Total: 1012
Fragments (<=2 words): 0  (0.0%)
Blobs (>=100 words):   0  (0.0%)

--- sentiment_informal ---
Total: 3536
Fragments (<=2 words): 1  (0.0%)
Blobs (>=100 words):   0  (0.0%)
Sample fragments: ['اخبار پڙهيم']



In [45]:
# ── Morpheme check, extended to all Stage 2 tokenizers ─────────────────────
ALL_MORPHEME_TOKENIZERS = {
    "SindhiLM_v1": tokenizers["SindhiLM_v1"],
    "SindhiLM_v3": tokenizers["SindhiLM_v3"],
    "FahadQazi_BPE": tokenizers["FahadQazi_BPE"],
    "XLM-RoBERTa": tokenizers["XLM-RoBERTa"],
    "mT5": tokenizers["mT5"],
    "Qwen2.5": tokenizers["Qwen2.5"],
    "BLOOM": tokenizers["BLOOM"],
    "Gemma3": tokenizers.get("Gemma3"),
    "Llama3.2": tokenizers.get("Llama3.2"),
}
# SindhiLM_v2 deliberately excluded — it's a confirmed duplicate of Qwen2.5,
# so its row would just repeat Qwen2.5's and add noise, not information.
ALL_MORPHEME_TOKENIZERS = {k: v for k, v in ALL_MORPHEME_TOKENIZERS.items() if v is not None}

rows = []
for category, words in MORPHEME_TEST_WORDS.items():
    for w in words:
        row = {"category": category, "word": w}
        for tok_name, tok in ALL_MORPHEME_TOKENIZERS.items():
            ids = tok.encode(w, add_special_tokens=False)
            row[tok_name] = decode_tokens_pretty(tok, ids)
        rows.append(row)

morpheme_df_stage2 = pd.DataFrame(rows)
morpheme_df_stage2.to_csv("sindhi_tokenizer_morpheme_check_stage2.csv", index=False)
morpheme_df_stage2

,category,word,SindhiLM_v1,SindhiLM_v3,FahadQazi_BPE,XLM-RoBERTa,mT5,Qwen2.5,BLOOM,Gemma3,Llama3.2
0,genitive_possessive,دنيا جو,▁ | د | ن | ي | ا | ▁ | ج | و,دنيا | ج | و,دنيا | جو,دنيا | جو,د | نيا | جو,دني | ا | ج | و,د | نيا | جو,دن | يا | جو,دن | يا | جو
1,genitive_possessive,خدا جو,▁ | خ | د | ا | ▁ | ج | و,خدا | ج | و,خدا | جو,خدا | جو,خدا | جو,خ | دا | ج | و,خدا | جو,خ | دا | جو,خ | دا | جو
2,genitive_possessive,پاڻي جي,▁ | <unk> | ا | ڻ | ي | ▁ | جي,پاڻي | ج | ي,پاڻي | جي,پاڻي | جي,پ | اڻي | جي,پ | ا | � | � | ي | ج | ي,پا | � | � | ي | جي,پا | ڻ | ي | جي,پ | ا | � | � | ي | جي
3,genitive_possessive,ماڻهن جو,▁ | ما | ڻ | ه | ن | ▁ | ج | و,ماڻهن | ج | و,ماڻهن | جو,ماڻهن | جو,ما | ڻ | هن | جو,ما | � | � | هن | ج | و,ما | � | � | هن | جو,ما | ڻ | هن | جو,ما | � | � | هن | جو
4,inflectional_suffixes,ڪتابن,▁ڪتاب | ن,ڪتابن,ڪتابن,ڪتاب | ن,ڪ | ت | ابن,ڪ | تاب | ن,� | � | تاب | ن,ڪ | تاب | ن,� | � | تاب | ن
5,inflectional_suffixes,شهرن,▁ | ش | ه | ر | ن,ش | ه | ر | ن,ش | ه | رن,شهر | ن,شهر | ن,شهر | ن,ش | هر | ن,شهر | ن,شهر | ن
6,inflectional_suffixes,توهانکي,▁ت | و | ه | ا | ن | ک | ي,ت | و | ه | ا | ن | کي,توهان | کي,توهان | کي,تو | هان | کي,تو | ه | ان | ک | ي,ت | وه | انک | ي,ت | وه | انک | ي,ت | وه | ان | ک | ي
7,inflectional_suffixes,منهنجو,▁ | م | ن | ه | ن | ج | و,م | نهن | ج | و,منهنجو,منهنجو,منه | نجو,من | ه | نج | و,منه | نج | و,من | ه | نج | و,من | ه | نج | و
8,inflectional_suffixes,هنجي,▁ | ه | ن | جي,ه | ن | ج | ي,هنجي,هن | جي,| ه | نجي,ه | نج | ي,ه | نج | ي,ه | نج | ي,ه | نج | ي
9,aspirated_digraphs,ٻار,▁ | ٻ | ا | ر,ٻار,ٻ | ار,ٻار,ٻ | ار,ٻ | ار,� | � | ار,ٻ | ار,� | � | ار


In [46]:
# ── Confirm whether the "�" tokens are genuine byte-fallback ──────────────
suspect_words = {
    "BLOOM": ("bigscience/bloom-560m", "ٻار"),
    "Llama3.2": ("meta-llama/Llama-3.2-1B", "ڳوٺ"),
    "Qwen2.5": ("Qwen/Qwen2.5-0.5B", "اسڪول"),
}

for name, (repo_id, word) in suspect_words.items():
    tok = tokenizers[name]
    ids = tok.encode(word, add_special_tokens=False)
    raw_tokens = tok.convert_ids_to_tokens(ids)
    print(f"\n{name} on '{word}':")
    print(f"  ids: {ids}")
    print(f"  raw tokens: {raw_tokens}")
    # If it's genuine byte-fallback, raw tokens will look like <0xNN> or
    # single-byte-looking strings rather than a decodable character


BLOOM on 'ٻار':
  ids: [151, 123, 713]
  raw tokens: ['Ù', '»', 'Ø§Ø±']

Llama3.2 on 'ڳوٺ':
  ids: [150, 111, 12942, 149, 118]
  raw tokens: ['Ú', '³', 'ÙĪ', 'Ù', 'º']

Qwen2.5 on 'اسڪول':
  ids: [91344, 145613, 72804]
  raw tokens: ['Ø§Ø³', 'Úª', 'ÙĪÙĦ']


In [49]:
# ── Corrected byte-fallback detection: works for SentencePiece <0xNN> AND
# GPT-2-style byte-level BPE tokens, by decoding each token ID individually
# rather than pattern-matching the token string ──────────────────────────

def compute_byte_fallback_rate(tokenizer, sentences, max_sentences=None):
    """A token counts as 'byte-fallback' if decoding it alone (not as part
    of its neighbors) yields a UTF-8 replacement character — meaning that
    token represents an incomplete multi-byte sequence, not a real character
    or subword unit."""
    sample = sentences if max_sentences is None else sentences[:max_sentences]
    total_tokens = 0
    fallback_tokens = 0
    for s in sample:
        ids = tokenizer.encode(s, add_special_tokens=False)
        for tid in ids:
            total_tokens += 1
            decoded_single = tokenizer.decode([tid])
            if "�" in decoded_single or decoded_single == "":
                fallback_tokens += 1
    rate = 100 * fallback_tokens / total_tokens if total_tokens else 0.0
    return rate, fallback_tokens, total_tokens

# Re-check every tokenizer with the corrected method
print(f"{'tokenizer':15s} {'byte_fallback_%':>16s} {'fallback/total':>16s}")
corrected_byte_fallback = {}
for name, tok in tokenizers.items():
    rate, n_fb, n_tot = compute_byte_fallback_rate(tok, wiki_sentences_clean, max_sentences=500)
    corrected_byte_fallback[name] = rate
    print(f"{name:15s} {rate:16.3f} {n_fb:>8d}/{n_tot:<8d}")

tokenizer        byte_fallback_%   fallback/total
SindhiLM_v1                0.000        0/35965   
SindhiLM_v2                7.253     1822/25119   
SindhiLM_v3                0.000        0/14073   
FahadQazi_BPE              7.990     1099/13754   
CustomSindhi_32k            0.108       11/10213   
CustomSindhi_64k            0.163       16/9807    
CustomSindhi_100k            0.156       15/9642    
CustomSindhi_150k            0.146       14/9557    
XLM-RoBERTa                1.408      181/12854   
mT5                       13.556     2576/19003   
Qwen2.5                    7.253     1822/25119   
BLOOM                     29.351     6284/21410   
Gemma3                     1.324      244/18426   
Llama3.2                  26.243     6420/24464   


In [50]:
# ── Corrected byte-fallback rate, ALL tokenizers × ALL 3 registers ─────────
ALL_REGISTERS_FOR_CHECK = {
    "flores200_anchor": flores_sentences,
    "wikipedia_formal": wiki_sentences_clean,
    "sentiment_informal": sentiment_sentences,
}

rows = []
for reg_name, sentences in ALL_REGISTERS_FOR_CHECK.items():
    print(f"\n=== {reg_name} ===")
    print(f"{'tokenizer':18s} {'byte_fallback_%':>16s} {'fallback/total':>16s}")
    for name, tok in tokenizers.items():
        rate, n_fb, n_tot = compute_byte_fallback_rate(tok, sentences, max_sentences=500)
        rows.append({"tokenizer": name, "register": reg_name,
                      "byte_fallback_pct_corrected": rate,
                      "fallback_tokens": n_fb, "total_tokens": n_tot})
        print(f"{name:18s} {rate:16.3f} {n_fb:>8d}/{n_tot:<8d}")

byte_fallback_full_df = pd.DataFrame(rows)
byte_fallback_full_df.to_csv("byte_fallback_corrected_all_registers.csv", index=False)

# Pivot for an easy side-by-side view
byte_fallback_pivot = byte_fallback_full_df.pivot(index="tokenizer", columns="register", values="byte_fallback_pct_corrected")
byte_fallback_pivot


=== flores200_anchor ===
tokenizer           byte_fallback_%   fallback/total
SindhiLM_v1                   0.000        0/53884   
SindhiLM_v2                   9.170     3562/38846   
SindhiLM_v3                   0.000        0/22764   
FahadQazi_BPE                 7.091     1416/19970   
CustomSindhi_32k              0.038        6/15600   
CustomSindhi_64k              0.113       17/15045   
CustomSindhi_100k             0.148       22/14850   
CustomSindhi_150k             0.122       18/14723   
XLM-RoBERTa                   1.369      268/19576   
mT5                          13.363     3868/28946   
Qwen2.5                       9.170     3562/38846   
BLOOM                        32.402    10866/33535   
Gemma3                        0.000        0/27346   
Llama3.2                     29.777    11402/38291   

=== wikipedia_formal ===
tokenizer           byte_fallback_%   fallback/total
SindhiLM_v1                   0.000        0/35965   
SindhiLM_v2                   7.

register,flores200_anchor,sentiment_informal,wikipedia_formal
tokenizer,,,
BLOOM,32.401968,31.997720,29.350771
CustomSindhi_100k,0.148148,0.096141,0.155569
CustomSindhi_150k,0.122258,0.110193,0.146489
CustomSindhi_32k,0.038462,0.094023,0.107706
CustomSindhi_64k,0.112994,0.095694,0.163149
FahadQazi_BPE,7.090636,5.843072,7.990403
Gemma3,0.000000,0.243170,1.324216
Llama3.2,29.777232,28.823978,26.242642
Qwen2.5,9.169541,9.048503,7.253473


In [52]:
# ── Sanity check: does compute_byte_fallback_rate actually detect fallback ──
# for SindhiLM_v1 / SindhiLM_v3, or is their 0.000% a blind spot like it was
# for XLM-R and the GPT-2-BPE tokenizers? Test with characters that are
# almost certainly OUTSIDE any Sindhi-trained vocab, so genuine fallback (or
# <unk>) MUST occur if the tokenizer/detector is working correctly.
STRESS_TEST_STRINGS = {
    "emoji": "😀🎉🔥",
    "chinese": "你好世界",
    "cyrillic": "привет мир",
    "rare_symbols": "∑∫√≠±∞",
}

for name in ["SindhiLM_v1", "SindhiLM_v3"]:
    tok = tokenizers[name]
    print(f"\n=== {name} ===")
    for label, text in STRESS_TEST_STRINGS.items():
        ids = tok.encode(text, add_special_tokens=False)
        raw_tokens = tok.convert_ids_to_tokens(ids)
        rate, n_fb, n_tot = compute_byte_fallback_rate(tok, [text])
        # Also check UNK rate on the same string, since v1/v3 might route
        # unknown content to <unk> instead of byte-fallback
        n_unk = sum(1 for t in raw_tokens if t == tok.unk_token)
        print(f"  {label:12s} raw_tokens={raw_tokens}")
        print(f"               byte_fallback_rate={rate:.1f}%  unk_count={n_unk}/{len(raw_tokens)}")


=== SindhiLM_v1 ===
  emoji        raw_tokens=['▁', '<unk>']
               byte_fallback_rate=0.0%  unk_count=1/2
  chinese      raw_tokens=['▁', '<unk>']
               byte_fallback_rate=0.0%  unk_count=1/2
  cyrillic     raw_tokens=['▁', '<unk>', '▁', '<unk>']
               byte_fallback_rate=0.0%  unk_count=2/4
  rare_symbols raw_tokens=['▁', '<unk>']
               byte_fallback_rate=0.0%  unk_count=1/2

=== SindhiLM_v3 ===
  emoji        raw_tokens=['[UNK]']
               byte_fallback_rate=0.0%  unk_count=1/1
  chinese      raw_tokens=['[UNK]']
               byte_fallback_rate=0.0%  unk_count=1/1
  cyrillic     raw_tokens=['[UNK]', '[UNK]']
               byte_fallback_rate=0.0%  unk_count=2/2
  rare_symbols raw_tokens=['[UNK]']
               byte_fallback_rate=0.0%  unk_count=1/1


In [51]:
# ── Corrected byte-fallback rate, full scale, all 3 registers ─────────────
byte_fallback_full = []
for name, tok in tokenizers.items():
    for reg_name, sentences in REGISTERS.items():
        rate, n_fb, n_tot = compute_byte_fallback_rate(tok, sentences, max_sentences=None)
        byte_fallback_full.append({
            "tokenizer": name,
            "register": reg_name,
            "byte_fallback_pct_corrected": rate,
            "fallback_tokens": n_fb,
            "total_tokens": n_tot,
        })
    # also check ByT5 separately since it's Tier C
rate, n_fb, n_tot = compute_byte_fallback_rate(byt5_tok, REGISTERS["sentiment_informal"], max_sentences=None)
print(f"ByT5-small (sentiment, sanity check): {rate:.3f}%  ({n_fb}/{n_tot})")

byte_fallback_df = pd.DataFrame(byte_fallback_full)
byte_fallback_df.to_csv("byte_fallback_corrected_full.csv", index=False)

# Merge the corrected column into results_df, replacing the old (broken) one
results_df = results_df.drop(columns=["byte_fallback_pct"], errors="ignore")
results_df = results_df.merge(
    byte_fallback_df[["tokenizer", "register", "byte_fallback_pct_corrected"]],
    on=["tokenizer", "register"], how="left"
)
results_df.to_csv("sindhi_tokenizer_benchmark_final_v2.csv", index=False)
byte_fallback_df.pivot(index="tokenizer", columns="register", values="byte_fallback_pct_corrected").round(2)

ByT5-small (sentiment, sanity check): 87.948%  (341179/387931)


register,flores200_anchor,sentiment_informal,wikipedia_formal
tokenizer,,,
BLOOM,32.70,32.65,29.96
CustomSindhi_100k,0.20,0.13,0.17
CustomSindhi_150k,0.20,0.14,0.18
CustomSindhi_32k,0.10,0.13,0.16
CustomSindhi_64k,0.17,0.13,0.17
FahadQazi_BPE,7.20,6.32,8.61
Gemma3,0.00,0.16,0.26
Llama3.2,30.01,29.38,26.72
Qwen2.5,9.22,8.97,7.50


In [53]:
# ── Fill final report gaps: full metrics on cleaned Wikipedia for all
# tokenizers + ByT5, since the earlier cleaned-Wikipedia run only printed
# fertility and single-token% ────────────────────────────────────────────

wiki_clean_full_results = []
for tok_name, tok in tokenizers.items():
    metrics = evaluate_tokenizer_on_sentences(tok, wiki_sentences_clean, MAX_SENTENCES_PER_REGISTER)
    if metrics is None:
        continue
    metrics["tokenizer"] = tok_name
    metrics["register"] = "wikipedia_formal_clean"
    metrics["vocab_size"] = tokenizer_vocab_sizes.get(tok_name)
    wiki_clean_full_results.append(metrics)
    print(f"{tok_name:15s} fertility={metrics['fertility']:.3f}  "
          f"compression={metrics['compression_chars_per_token']:.3f}  "
          f"single_token%={metrics['single_token_pct']:.2f}  "
          f"unk%={metrics['unk_rate_pct']:.3f}")

# ByT5 on cleaned Wikipedia — same treatment, kept separate as Tier C
byt5_wiki_clean_metrics = evaluate_tokenizer_on_sentences(byt5_tok, wiki_sentences_clean, MAX_SENTENCES_PER_REGISTER)
byt5_wiki_clean_metrics["tokenizer"] = "ByT5-small"
byt5_wiki_clean_metrics["register"] = "wikipedia_formal_clean"
print(f"\nByT5-small      fertility={byt5_wiki_clean_metrics['fertility']:.3f}  "
      f"compression={byt5_wiki_clean_metrics['compression_chars_per_token']:.3f}  "
      f"single_token%={byt5_wiki_clean_metrics['single_token_pct']:.2f}  "
      f"unk%={byt5_wiki_clean_metrics['unk_rate_pct']:.3f}")

wiki_clean_full_df = pd.DataFrame(wiki_clean_full_results)
wiki_clean_full_df.to_csv("wikipedia_clean_full_metrics.csv", index=False)
wiki_clean_full_df

SindhiLM_v1     fertility=4.271  compression=1.106  single_token%=0.18  unk%=15.137
SindhiLM_v2     fertility=2.568  compression=1.840  single_token%=25.49  unk%=0.000
SindhiLM_v3     fertility=1.726  compression=2.737  single_token%=55.42  unk%=4.949
FahadQazi_BPE   fertility=2.002  compression=2.360  single_token%=37.38  unk%=0.000
CustomSindhi_32k fertility=1.249  compression=3.781  single_token%=84.03  unk%=0.000
CustomSindhi_64k fertility=1.194  compression=3.957  single_token%=86.89  unk%=0.000
CustomSindhi_100k fertility=1.171  compression=4.033  single_token%=87.98  unk%=0.000
CustomSindhi_150k fertility=1.157  compression=4.084  single_token%=88.58  unk%=0.000
XLM-RoBERTa     fertility=1.589  compression=2.972  single_token%=63.14  unk%=0.111
mT5             fertility=2.285  compression=2.067  single_token%=22.08  unk%=0.077
Qwen2.5         fertility=2.568  compression=1.840  single_token%=25.49  unk%=0.000
BLOOM           fertility=2.569  compression=1.839  single_token%=25.3

,n_sentences,n_words,n_tokens,fertility,compression_chars_per_token,single_token_pct,unk_rate_pct,byte_fallback_pct,tokenizer,register,vocab_size
0,25480,435947,1861974,4.271102,1.105940,0.180527,15.137215,0.000000,SindhiLM_v1,wikipedia_formal_clean,43
1,25480,435947,1119363,2.567658,1.839646,25.492090,0.000000,0.000000,SindhiLM_v2,wikipedia_formal_clean,151643
2,25480,435947,752322,1.725719,2.737168,55.415222,4.949343,0.000000,SindhiLM_v3,wikipedia_formal_clean,32000
3,25480,435947,872655,2.001746,2.359732,37.382985,0.000000,0.000000,FahadQazi_BPE,wikipedia_formal_clean,13031
4,25480,435947,544673,1.249402,3.780676,84.030398,0.000000,0.018543,CustomSindhi_32k,wikipedia_formal_clean,32000
5,25480,435947,520417,1.193762,3.956888,86.887167,0.000000,0.019408,CustomSindhi_64k,wikipedia_formal_clean,64000
6,25480,435947,510651,1.171360,4.032562,87.978355,0.000000,0.019779,CustomSindhi_100k,wikipedia_formal_clean,100000
7,25480,435947,504262,1.156705,4.083655,88.580722,0.000000,0.020029,CustomSindhi_150k,wikipedia_formal_clean,150000
8,25480,435947,692771,1.589117,2.972457,63.138638,0.111437,0.000000,XLM-RoBERTa,wikipedia_formal_clean,250002
9,25480,435947,996189,2.285115,2.067110,22.078601,0.076692,0.000000,mT5,wikipedia_formal_clean,250100


In [54]:
# ── Final pre-publication check: how does each tokenizer actually generate ──
# tokens, and how many? Full stats + concrete examples, one tokenizer at a
# time, across a shared sample sentence set (same sentences for every
# tokenizer, so counts are directly comparable).
import numpy as np

CHECK_SAMPLE_SENTENCES = wiki_sentences_clean[:200]  # same 200 sentences for every tokenizer
EXAMPLE_SENTENCES = wiki_sentences_clean[:3]          # a few to show token-by-token

summary_rows = []
for name, tok in tokenizers.items():
    token_counts = []
    for s in CHECK_SAMPLE_SENTENCES:
        ids = tok.encode(s, add_special_tokens=False)
        token_counts.append(len(ids))
    token_counts = np.array(token_counts)

    summary_rows.append({
        "tokenizer": name,
        "vocab_size": tokenizer_vocab_sizes.get(name),
        "total_tokens_200_sent": int(token_counts.sum()),
        "avg_tokens_per_sentence": round(token_counts.mean(), 2),
        "min_tokens": int(token_counts.min()),
        "max_tokens": int(token_counts.max()),
        "median_tokens": int(np.median(token_counts)),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("avg_tokens_per_sentence")
summary_df.to_csv("final_token_generation_summary.csv", index=False)
print(summary_df.to_string(index=False))

# Concrete worked examples: full tokenization of the same 3 sentences,
# for every tokenizer, so you can visually inspect actual behavior
print("\n" + "="*80)
print("WORKED EXAMPLES — same 3 sentences, every tokenizer")
print("="*80)
for s in EXAMPLE_SENTENCES:
    print(f"\nSentence: {s}")
    for name, tok in tokenizers.items():
        ids = tok.encode(s, add_special_tokens=False)
        pieces = tok.convert_ids_to_tokens(ids)
        print(f"  {name:18s} ({len(pieces):3d} tokens): {pieces}")

        tokenizer  vocab_size  total_tokens_200_sent  avg_tokens_per_sentence  min_tokens  max_tokens  median_tokens
CustomSindhi_150k      150000                   3710                    18.55           3          83             16
CustomSindhi_100k      100000                   3742                    18.71           3          83             17
 CustomSindhi_64k       64000                   3823                    19.11           3          87             17
 CustomSindhi_32k       32000                   4011                    20.06           3          89             17
      XLM-RoBERTa      250002                   4972                    24.86           3         104             22
      SindhiLM_v3       32000                   5240                    26.20           4         119             22
    FahadQazi_BPE       13031                   5688                    28.44           3         128             25
              mT5      250100                   7248            